# Sentence Transformer + Logistic Regression

Tạo embedding bằng mô hình pretrained `sentence-transformers/all-MiniLM-L6-v2`, sau đó so sánh hai Logistic Regression trên Validation. Notebook dùng nguyên các CSV hiện tại; không chia lại, không ghi đè dữ liệu và không đánh giá/chọn mô hình trên Test. Việc tải mô hình pretrained cần kết nối Internet ở lần chạy đầu tiên.

## 1. Giải thích thay đổi số lượng mẫu
Ở lần chia dữ liệu trước, 9.763 hàng ban đầu được gom theo nội dung review không phân biệt hoa/thường: còn 7.964 nội dung duy nhất. Có 18 nhóm (234 hàng) mang nhãn mâu thuẫn nên bị loại. Trong 9.529 hàng còn lại, 1.583 hàng là review lặp; giữ một bản cho mỗi nội dung nhất quán, còn 7.946 mẫu.

Do đó: 9.763 − 234 hàng thuộc nhóm nhãn mâu thuẫn − 1.583 bản lặp = **7.946**. Đây là các bước làm sạch đã ghi nhận, không phải mất dữ liệu ngoài dự kiến. Ô dưới đối chiếu tổng và số mẫu các CSV hiện tại; không tái tạo hay ghi lại chúng.

In [1]:
from pathlib import Path
from time import perf_counter
import os

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_recall_fscore_support, precision_score, recall_score,
)

DATA_DIR = Path('data/processed')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data/processed')

# Đọc nguyên trạng các tệp hiện có; không fit hoặc đánh giá trên Test.
train_df = pd.read_csv(DATA_DIR / 'train.csv')
validation_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
splits = {'train': train_df, 'validation': validation_df, 'test': test_df}
expected_sizes = {'train': 5562, 'validation': 1192, 'test': 1192}
for name, frame in splits.items():
    assert len(frame) == expected_sizes[name], f'{name}: số hàng không khớp'
    assert list(frame.columns) == ['review', 'label']
    assert frame['review'].notna().all()
    assert set(frame['label'].unique()).issubset({0, 1})

def review_key(text):
    return str(text).casefold().strip()

review_keys = {name: set(frame['review'].map(review_key)) for name, frame in splits.items()}
assert not (review_keys['train'] & review_keys['validation'])
assert not (review_keys['train'] & review_keys['test'])
assert not (review_keys['validation'] & review_keys['test'])

print('Số mẫu theo tập:', {name: len(frame) for name, frame in splits.items()})
print('Tổng:', sum(map(len, splits.values())))
print('Các tập có nội dung review trùng nhau: không')
print('Tỷ lệ lớp (0, 1):', {name: frame['label'].value_counts(normalize=True).sort_index().round(4).to_dict()
                            for name, frame in splits.items()})

Số mẫu theo tập: {'train': 5562, 'validation': 1192, 'test': 1192}
Tổng: 7946
Các tập có nội dung review trùng nhau: không
Tỷ lệ lớp (0, 1): {'train': {0: 0.0708, 1: 0.9292}, 'validation': {0: 0.0705, 1: 0.9295}, 'test': {0: 0.0713, 1: 0.9287}}


## 2. Tạo embedding trên CPU
Nạp mô hình Sentence Transformer pretrained. Chuyển Train và Validation thành vector 384 chiều. Test được đọc/kiểm tra tính nguyên trạng nhưng không được encode hoặc dùng trong đánh giá. Thời gian embedding được đo tách riêng cho Train và Validation; thời gian tải mô hình được báo cáo riêng.

In [2]:
torch.set_num_threads(min(4, os.cpu_count() or 1))
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
load_start = perf_counter()
sentence_model = SentenceTransformer(model_name, device='cpu')
model_load_seconds = perf_counter() - load_start

train_start = perf_counter()
X_train_embeddings = sentence_model.encode(
    train_df['review'].tolist(), batch_size=32, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=False,
)
train_embedding_seconds = perf_counter() - train_start

validation_start = perf_counter()
X_validation_embeddings = sentence_model.encode(
    validation_df['review'].tolist(), batch_size=32, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=False,
)
validation_embedding_seconds = perf_counter() - validation_start

y_train = train_df['label'].to_numpy()
y_validation = validation_df['label'].to_numpy()
print(f'Thiết bị: CPU | Kích thước embedding: {X_train_embeddings.shape[1]}')
print(f'Thời gian tải mô hình: {model_load_seconds:.2f} giây')
print(f'Thời gian embedding Train ({len(y_train):,} review): {train_embedding_seconds:.2f} giây')
print(f'Thời gian embedding Validation ({len(y_validation):,} review): {validation_embedding_seconds:.2f} giây')
print('Hình dạng embedding:', X_train_embeddings.shape, X_validation_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\ADMIN\Documents\Elden-Ring-Sentiment-Analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Thiết bị: CPU | Kích thước embedding: 384
Thời gian tải mô hình: 17.35 giây
Thời gian embedding Train (5,562 review): 20.25 giây
Thời gian embedding Validation (1,192 review): 4.09 giây
Hình dạng embedding: (5562, 384) (1192, 384)


## 3. Huấn luyện và đánh giá trên Validation
Cả hai cấu hình dùng cùng embedding. `max_iter=1000` giúp Logistic Regression có đủ vòng lặp hội tụ; cấu hình thứ hai thêm `class_weight='balanced'`. Thời gian train không bao gồm thời gian tạo embedding. Ma trận nhầm lẫn sắp theo nhãn 0 rồi 1.

In [3]:
model_configs = {
    'Mặc định': LogisticRegression(max_iter=1000),
    'class_weight=balanced': LogisticRegression(max_iter=1000, class_weight='balanced'),
}
results = []
confusion_matrices = {}
trained_models = {}

for name, classifier in model_configs.items():
    start = perf_counter()
    classifier.fit(X_train_embeddings, y_train)
    training_seconds = perf_counter() - start
    predictions = classifier.predict(X_validation_embeddings)
    class_precision, class_recall, class_f1, _ = precision_recall_fscore_support(
        y_validation, predictions, labels=[0, 1], zero_division=0
    )
    results.append({
        'Cấu hình': name,
        'Accuracy': accuracy_score(y_validation, predictions),
        'Precision (lớp 1)': precision_score(y_validation, predictions, pos_label=1, zero_division=0),
        'Recall (lớp 1)': recall_score(y_validation, predictions, pos_label=1, zero_division=0),
        'F1 (lớp 1)': f1_score(y_validation, predictions, pos_label=1, zero_division=0),
        'Macro-F1': f1_score(y_validation, predictions, average='macro', zero_division=0),
        'Precision (lớp 0)': class_precision[0],
        'Recall (lớp 0)': class_recall[0],
        'F1 (lớp 0)': class_f1[0],
        'Thời gian train (giây)': training_seconds,
    })
    confusion_matrices[name] = confusion_matrix(y_validation, predictions, labels=[0, 1])
    trained_models[name] = classifier

results_table = pd.DataFrame(results).set_index('Cấu hình')
display(results_table.round(4))
for name, matrix in confusion_matrices.items():
    print(f'Ma trận nhầm lẫn — {name} (hàng = nhãn thật, cột = dự đoán):')
    display(pd.DataFrame(matrix, index=['Thật 0', 'Thật 1'],
                         columns=['Dự đoán 0', 'Dự đoán 1']))

,Accuracy,Precision (lớp 1),Recall (lớp 1),F1 (lớp 1),Macro-F1,Precision (lớp 0),Recall (lớp 0),F1 (lớp 0),Thời gian train (giây)
Cấu hình,,,,,,,,,
Mặc định,0.9421,0.9421,0.9991,0.9698,0.6433,0.9412,0.1905,0.3168,0.0297
class_weight=balanced,0.8423,0.9802,0.8475,0.9090,0.6589,0.2778,0.7738,0.4088,0.0146


Ma trận nhầm lẫn — Mặc định (hàng = nhãn thật, cột = dự đoán):


,Dự đoán 0,Dự đoán 1
Thật 0,16,68
Thật 1,1,1107


Ma trận nhầm lẫn — class_weight=balanced (hàng = nhãn thật, cột = dự đoán):


,Dự đoán 0,Dự đoán 1
Thật 0,65,19
Thật 1,169,939


## 4. So sánh với TF-IDF + Logistic Regression
Các chỉ số TF-IDF bên dưới lấy từ kết quả Validation đã ghi trong `03_logistic_regression.ipynb`, trên cùng các phân hoạch hiện tại. Test không được dùng trong so sánh này.

In [4]:
comparison_rows = []
tfidf_results = {
    'Mặc định': {'Accuracy': 0.9304, 'Macro-F1': 0.5049, 'F1 lớp 0': 0.0460},
    'class_weight=balanced': {'Accuracy': 0.8842, 'Macro-F1': 0.6827, 'F1 lớp 0': 0.4298},
}
for config, scores in tfidf_results.items():
    comparison_rows.append({'Vector hóa': 'TF-IDF', 'Cấu hình': config, **scores})
for config, row in results_table.iterrows():
    comparison_rows.append({
        'Vector hóa': 'Sentence Transformer', 'Cấu hình': config,
        'Accuracy': row['Accuracy'], 'Macro-F1': row['Macro-F1'],
        'F1 lớp 0': row['F1 (lớp 0)'],
    })
comparison = pd.DataFrame(comparison_rows).set_index(['Vector hóa', 'Cấu hình'])
display(comparison.round(4))

best_config = results_table['Macro-F1'].idxmax()
print(f"Sentence Transformer có Macro-F1 cao nhất với cấu hình: {best_config} ({results_table.loc[best_config, 'Macro-F1']:.4f}).")
print('Đây chỉ là so sánh trên Validation; chưa đánh giá trên Test.')

Accuracy  Macro-F1  F1 lớp 0
Vector hóa           Cấu hình                                           
TF-IDF               Mặc định                 0.9304    0.5049    0.0460
                     class_weight=balanced    0.8842    0.6827    0.4298
Sentence Transformer Mặc định                 0.9421    0.6433    0.3168
                     class_weight=balanced    0.8423    0.6589    0.4088

Sentence Transformer có Macro-F1 cao nhất với cấu hình: class_weight=balanced (0.6589).
Đây chỉ là so sánh trên Validation; chưa đánh giá trên Test.
